In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg, col,monotonically_increasing_id 

spark = SparkSession.builder.appName("GoldLayerCreation").getOrCreate()

# Read the necessary Silver tables
silver_users = spark.read.table("ecommerce_fashion.silver.users")
silver_buyers = spark.read.table("ecommerce_fashion.silver.buyers")
silver_sellers = spark.read.table("ecommerce_fashion.silver.sellers")
silver_countries = spark.read.table("ecommerce_fashion.silver.countries")

In [0]:
#There are common column names in these four tables all corressponding to different values and elements. Hence we need to first find out which are these columns and then rename them
#make three lists called users_cols, buyers_cols and so on, containing all the column names of all the tables. We will then use set operations to find the columns with the same names
users_cols = silver_users.columns
buyers_cols = silver_buyers.columns
sellers_cols = silver_sellers.columns
countries_cols = silver_countries.columns

all_combined = [users_cols, buyers_cols, sellers_cols, countries_cols]

combinations = {
    '0,1' : "user_buyer",
    '0,2' : "user_seller",
    '0,3' : "user_country",
    '1,2' : "buyer_seller",
    '1,3' : "buyer_country",
    '2,3' : 'seller_country'
}

common_elements = {}

common_intersection = set.intersection(set(users_cols), set(buyers_cols), set(sellers_cols), set(countries_cols))

for i in range(0, len(all_combined)):
    for j in range(i+1, len(all_combined)):
        table_name = combinations[f'{i},{j}']
        common_elements[table_name] = set.intersection(set(all_combined[i]), set(all_combined[j]))

print(common_elements)
        








In [0]:
#Here we will be doing our renaming. 
def rename(df, value, key):
    df = df.withColumnRenamed(value, value + "_" + key)

    return df

for i,j in common_elements.items():
    if i == 'user_buyer':
        for k in j:
            a = 'user'
            b = 'buyer'
            if (k != "country"):
                silver_users = rename(silver_users, k, a)
                silver_buyers = rename(silver_buyers, k, b)
    if i == 'user_seller':
        for k in j:
            a = 'user'
            b = 'seller'
            if (k != "country"):
                silver_users = rename(silver_users, k, a)
                silver_sellers = rename(silver_sellers, k, b)
    if i == 'user_country':
        for k in j:
            a = 'user'
            b = 'country'
            if (k != "country"):
                silver_users = rename(silver_users, k, a)
                silver_countries = rename(silver_countries, k, b)
    if i == 'buyer_seller':
        for k in j:
            a = 'buyer'
            b = 'seller'
            if (k != "country"):
                silver_buyers = rename(silver_buyers, k, a)
                silver_sellers = rename(silver_sellers, k, b)
    if i == 'buyer_country':
        for k in j:
            a = 'buyer'
            b = 'country'
            if (k != "country"):
                silver_buyers = rename(silver_buyers, k, a)
                silver_country = rename(silver_countries, k, b)
    if i == 'seller_country':
        for k in j:
            a = 'seller'
            b = 'country'
            if (k != "country"):
                silver_sellers = rename(silver_sellers, k, a)
                silver_country = rename(silver_countries, k, b)

    
print(silver_users.printSchema())
print(silver_buyers.printSchema())
print(silver_sellers.printSchema())
print(silver_countries.printSchema())



In [0]:
comprehensive_user_table = silver_users \
    .join(silver_countries, ["country"], "outer") \
    .join(silver_buyers, ["country"], "outer") \
    .join(silver_sellers, ["country"], "outer")

comprehensive_user_table.show(5)

In [0]:
comprehensive_user_table.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.gold.comprehensive_table")